In [1]:
# ! ls -lah /kg_graphrag

In [2]:
# ! pip3 install elasticsearch

In [70]:
from elasticsearch import AsyncElasticsearch
from pprint import pprint
import json
import asyncio
from tqdm.asyncio import tqdm
import pandas as pd
import os

In [16]:
index_name = "9e4f0121-b7c4-4087-9995-4af351c58ef5"
output_file=f"es_export-col_{index_name}.json"

# Export data from ES for tatneft KG experiment

In [14]:
async def export_elasticsearch_data(client: AsyncElasticsearch, index_name: str, output_file: str, batch_size:int=1000):
    """
    Export all data from Elasticsearch index to a file using async API.
    
    Args:
        index_name: Name of the Elasticsearch index
        output_file: Path to output file
        batch_size: Number of documents to retrieve per batch
    """
    try:
        # Get total document count for progress tracking
        count_resp = await client.count(index=index_name)
        total_docs = count_resp["count"]
        print(f"Exporting {total_docs} documents from index '{index_name}'")
        
        # Initialize scroll
        resp = await client.search(
            index=index_name,
            scroll="5m",  # Keep the search context alive for 5 minutes
            size=batch_size,
            body={"query": {"match_all": {}}}
        )
        
        scroll_id = resp["_scroll_id"]
        hits = resp["hits"]["hits"]
        
        # Write to file
        with open(output_file, "w") as f:
            # Process initial batch
            for doc in hits:
                f.write(json.dumps(doc) + "\n")
            
            # Process remaining batches with progress bar
            with tqdm(total=total_docs, initial=len(hits)) as pbar:
                while len(hits) > 0:
                    # Get next batch of results
                    resp = await client.scroll(
                        scroll_id=scroll_id,
                        scroll="5m"
                    )
                    
                    # Update scroll_id
                    scroll_id = resp["_scroll_id"]
                    hits = resp["hits"]["hits"]
                    
                    # Write batch to file
                    for doc in hits:
                        f.write(json.dumps(doc) + "\n")
                    
                    # Update progress bar
                    pbar.update(len(hits))
        
        print(f"Export completed successfully. Data saved to {output_file}")
    
    except Exception as e:
        print(f"Error exporting data: {str(e)}")
        raise
    
    finally:
        # Clear the scroll context to free resources
        if 'scroll_id' in locals() and scroll_id:
            await client.clear_scroll(scroll_id=scroll_id)
        
        # Close the client
        await client.close()

In [15]:
es = AsyncElasticsearch(
    hosts="http://d.dgx:9202",
    basic_auth=("elastic", "admin")
)


await export_elasticsearch_data(client=es, index_name=index_name, output_file=output_file)

Exporting 38404 documents from index '9e4f0121-b7c4-4087-9995-4af351c58ef5'


/tmp/ipykernel_2792/2207651971.py:17: DeprecationWarning: Received 'size' via a specific parameter in the presence of a 'body' parameter, which is deprecated and will be removed in a future version. Instead, use only 'body' or only specific parameters.
  resp = await client.search(
100%|██████████| 38404/38404 [03:09<00:00, 196.92it/s]


Export completed successfully. Data saved to es_export-col_9e4f0121-b7c4-4087-9995-4af351c58ef5.json


# Extract limited set of chunks and make a single text of it for GraphRAG

In [54]:
with open(output_file, "r") as f:
    records = (json.loads(line) for line in f.readlines())
    data = [
        {
            "_id": r["_id"],
            "paragraph": r["_source"]["paragraph"],
            "filename": r["_source"]["metadata"].get("file_name", None),
            "chapter": r["_source"]["metadata"].get("chapter", None),
            "chapted_id": r["_source"]["metadata"].get("chapter_id", None),
            "chunk_id": r["_source"]["metadata"].get("chunk_id", None),
        }
        for r in records
    ]

df = pd.DataFrame(data)

In [55]:
df

,_id,paragraph,filename,chapter,chapted_id,chunk_id
0,c259338d-a0d0-4583-a2bf-ccfd5e5f8184,"Р. X. MfOHMMOB, А. М. lUaBamMB, Р. Б. Хисамов,...","53 ÐÐµÐ¾Ð»Ð¾Ð³Ð¸Ñ,ÑÐ°Ð·ÑÐ°Ð±Ð¾ÑÐºÐ° Ð¸ Ñ...",,1,1
1,03f31a96-0d8b-44f4-a6b2-15c005130569,"водств, а также аспирантов и студентов нефтяны...","53 ÐÐµÐ¾Ð»Ð¾Ð³Ð¸Ñ,ÑÐ°Ð·ÑÐ°Ð±Ð¾ÑÐºÐ° Ð¸ Ñ...",,1,2
2,1ec3fa0f-9163-4537-8be7-2bd38ccd0ff7,АЗН — активные запасы нефти АКЦ — акустический...,"53 ÐÐµÐ¾Ð»Ð¾Ð³Ð¸Ñ,ÑÐ°Ð·ÑÐ°Ð±Ð¾ÑÐºÐ° Ð¸ Ñ...",АББРЕВИАТУРА,2,3
3,30aa10fb-194c-4c8b-9d96-71fa52b12d7d,СКГ — сернокислый глинозем СКО — соляно-кислот...,"53 ÐÐµÐ¾Ð»Ð¾Ð³Ð¸Ñ,ÑÐ°Ð·ÑÐ°Ð±Ð¾ÑÐºÐ° Ð¸ Ñ...",АББРЕВИАТУРА,2,4
4,73686d68-6d64-4c97-96bd-26103f216a7c,На месторождении в отложениях бобриковского го...,"53 ÐÐµÐ¾Ð»Ð¾Ð³Ð¸Ñ,ÑÐ°Ð·ÑÐ°Ð±Ð¾ÑÐºÐ° Ð¸ Ñ...",7. ПРОЕКТИРОВАНИЕ И АНАЛИЗ РАЗРАБОТКИ ЗАЛЕЖЕЙ ...,3,5
...,...,...,...,...,...,...
38399,993873d8-db05-4ea7-a97e-c5b12c758645,совместное применение АСК и ПАВ позволяет повы...,25 ÐÐ°ÑÑÐ½ÑÐµ ÑÑÑÐ´Ñ ÐÑÑÐ»Ð¸Ð¼Ð¾Ð² ...,МЕТОДОВ УВЕЛИЧЕНИЯ НЕФТЕОТДАЧИ ПЛАСТОВ НА МЕСТ...,117,996
38400,6130a316-7e9d-4fe4-af70-2c29977c428f,"Проведенный объем теоретических, экспериментал...",25 ÐÐ°ÑÑÐ½ÑÐµ ÑÑÑÐ´Ñ ÐÑÑÐ»Ð¸Ð¼Ð¾Ð² ...,МЕТОДОВ УВЕЛИЧЕНИЯ НЕФТЕОТДАЧИ ПЛАСТОВ НА МЕСТ...,117,997
38401,2c64d534-e076-45fe-8dcc-92908535aa6b,- сравнение показателей разработки сходных по ...,25 ÐÐ°ÑÑÐ½ÑÐµ ÑÑÑÐ´Ñ ÐÑÑÐ»Ð¸Ð¼Ð¾Ð² ...,МЕТОДОВ УВЕЛИЧЕНИЯ НЕФТЕОТДАЧИ ПЛАСТОВ НА МЕСТ...,117,998
38402,2e90f787-e470-4014-a7b3-949a04a5bf6a,"По состоянию на 01.01.1996 г., по все.м тсхнол...",25 ÐÐ°ÑÑÐ½ÑÐµ ÑÑÑÐ´Ñ ÐÑÑÐ»Ð¸Ð¼Ð¾Ð² ...,МЕТОДОВ УВЕЛИЧЕНИЯ НЕФТЕОТДАЧИ ПЛАСТОВ НА МЕСТ...,117,999


In [56]:
fc_df = df[["filename", "chapter"]].drop_duplicates()
fc_df = fc_df.groupby("filename").count().reset_index()
fc_df = fc_df.sort_values("chapter").reset_index(drop=True)
fc_df

,filename,chapter
0,14 ÐÐµÐ¾Ð»Ð¾Ð³Ð¸ÑÐµÑÐºÐ°Ñ Ð¾ÑÐµÐ½ÐºÐ° Ð¿Ð...,5
1,79 ÐÐ¸Ð´ÑÐ¾Ð´Ð¸Ð½Ð°Ð¼Ð¸ÑÐµÑÐºÐ¸Ðµ Ð¼ÐµÑÐ¾...,12
2,66 Ð¡Ð¾ÑÑÐ¾ÑÐ½Ð¸Ðµ Ð¸ Ð¿ÑÑÐ¸ Ð¿Ð¾Ð²ÑÑÐµ...,13
3,2 ÐÑÑÐ°Ð±Ð¾ÑÐºÐ° ÑÑÐµÑÐ¸Ð½Ð¾Ð²Ð°ÑÐ¾-Ð¿...,19
4,78 ÐÐ½ÑÐµÑÐ¿ÑÐµÑÐ°ÑÐ¸Ñ ÑÐµÐ·ÑÐ»ÑÑÐ°...,19
...,...,...
86,60 ÐÐµÐ¾Ð»Ð¾Ð³Ð¾ÑÐ°Ð·Ð²ÐµÐ´Ð¾ÑÐ½ÑÐµ ÑÐ°Ð±...,229
87,46 Ð¡Ð¾Ð²ÑÐµÐ¼ÐµÐ½Ð½ÑÐµ Ð¼ÐµÑÐ¾Ð´Ñ ÑÐ¿ÑÐ...,232
88,45 Ð¡Ð¾Ð²ÑÐµÐ¼ÐµÐ½Ð½ÑÐµ Ð¼ÐµÑÐ¾Ð´Ñ Ð¿Ð¾Ð²Ñ...,233
89,58 Ð£Ð²ÐµÐ»Ð¸ÑÐµÐ½Ð¸Ðµ Ð¾Ñ Ð²Ð°ÑÐ° Ð¿ÑÐ¾Ð´Ñ...,242


In [57]:
doc_name = fc_df.iloc[3]["filename"]

In [65]:
doc_df = df[df["filename"] == doc_name].sort_values(["filename", "chapted_id", "chunk_id"])
text = "".join(doc_df["paragraph"].tolist())
with open("tat_doc.txt", "w") as f:
    f.write(text)

# Invetigate created graph

In [83]:
kg_base_path = "/kg_graphrag/graphs/graphrag_data/"

rels_path = os.path.join(kg_base_path, "output", "relationships.parquet")
entities_path = os.path.join(kg_base_path, "output", "entities.parquet")
communities_path = os.path.join(kg_base_path, "output", "communities.parquet")
community_reports_path = os.path.join(kg_base_path, "output", "community_reports.parquet")
text_units_path = os.path.join(kg_base_path, "output", "text_units.parquet")

In [84]:
! ls -lah {kg_base_path}/output/

total 6.3M
drwxr-xr-x 3 root root 4.0K Mar 18 12:39 .
drwxrwxr-x 7 7003 7003 4.0K Mar 18 12:27 ..
-rw-r--r-- 1 root root  72K Mar 18 12:35 communities.parquet
-rw-r--r-- 1 root root 461K Mar 18 12:39 community_reports.parquet
-rw-r--r-- 1 root root    2 Mar 18 12:39 context.json
-rw-r--r-- 1 root root  93K Mar 18 12:27 documents.parquet
-rw-r--r-- 1 root root 605K Mar 18 12:39 embeddings.community.full_content.parquet
-rw-r--r-- 1 root root 4.2M Mar 18 12:39 embeddings.entity.description.parquet
-rw-r--r-- 1 root root 342K Mar 18 12:39 embeddings.text_unit.text.parquet
-rw-r--r-- 1 root root  95K Mar 18 12:35 entities.parquet
-rw-r--r-- 1 root root  99K Mar 18 12:35 graph.graphml
drwxr-xr-x 5 root root 4.0K Mar 18 12:39 lancedb
-rw-r--r-- 1 root root 148K Mar 18 12:35 relationships.parquet
-rw-r--r-- 1 root root  816 Mar 18 12:39 stats.json
-rw-r--r-- 1 root root 166K Mar 18 12:35 text_units.parquet


In [79]:
rels_df = pd.read_parquet(rels_path)
rels_df[["source", "target", "description", "weight"]]

In [81]:
ents_df = pd.read_parquet(entities_path)
ents_df

In [ ]:
communities_df = pd.read_parquet(communities_path)
communities_df

In [ ]:
community_reports_df = pd.read_parquet(community_reports_path)
community_reports_df

In [105]:
community_reports_df["level"].unique()

array([2, 1, 0])

In [112]:
row = community_reports_df[community_reports_df['level'] == 2].iloc[2]

In [113]:
row["title"]

'Regulatory Compliance in Financial Institutions'

In [114]:
row["summary"]

'This community focuses on the regulatory compliance requirements for financial institutions, particularly those recognized by regulatory authorities. Key entities include Rule 10.3.2(3), Rule 10.4.1, and Recognised Bodies, which are subject to stringent AML/TFS requirements. The relationships highlight the importance of customer due diligence, regular reviews of AML policies, and compliance with wire transfer regulations.'

In [ ]:
text_units_df = pd.read_parquet(text_units_path)
text_units_df